---
image: example.gif
execute: 
  enabled: true
---

# Using the VidigiPriorityStore to Simplify Model Code Changes with Priority Resources

In vidigi 1.0.0, a new version of VidigiPriorityStore has been added that can be used almost exactly like a resource - reducing the amount of rewriting required to incorporate vidigi into your model. 

This allows us to use the pattern

```python
with self.treatment_cubicles.request(priority=patient.priority) as req:

    treatment_resource = yield req

    ### Continue all code in the indented portion that requires the resource, with the resource
    ### automatically being returned to the store when the indented portion completes
```

Instead of 

```python
treatment_resource = yield self.treatment_cubicles.get(priority=patient.priority)

### Continue all code that requires the resource

self.treatment_cubicles.put(treatment_resource)
```

(even though we are still using a Store behind the scenes)


This minimizes the syntax changes and rewriting that are required when converting an existing model built using resources to a vidigi-compatible state.

When setting up the resources, we simply use the pattern

```
from vidigi.utils import VidigiPriorityStore, populate_store

...

self.treatment_cubicles = VidigiPriorityStore(self.env)

populate_store(num_resources=g.n_cubicles, # or wherever you are storing your resource counts
               simpy_store=self.treatment_cubicles, # pass in the VidgiStore we created
               sim_env=self.env # include the simpy env this will sit in
               )
```

In [ ]:
from ex_7_model_classes import Trial, g
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import generate_animation, animate_activity_log
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"
import os

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "ex_7_model_classes.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
my_trial = Trial()

my_trial.run_trial()

In [ ]:
my_trial.all_event_logs.head(50)

In [ ]:
STEP_SNAPSHOT_MAX = 45
LIMIT_DURATION = g.sim_duration
WRAP_QUEUES_AT = 15

In [ ]:
full_patient_df = reshape_for_animations(
    event_log=my_trial.all_event_logs[my_trial.all_event_logs['run']==1],
    every_x_time_units=2,
    entity_col_name="patient",
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    limit_duration=LIMIT_DURATION,
    debug_mode=True
    )

full_patient_df.head(15)

In [ ]:
event_position_df = pd.DataFrame([
                    {'event': 'arrival',
                     'x':  50, 'y': 300,
                     'label': "Arrival" },

                    # Triage - minor and trauma
                    {'event': 'treatment_wait_begins',
                     'x':  205, 'y': 275,
                     'label': "Waiting for Treatment"},

                    {'event': 'treatment_begins',
                     'x':  205, 'y': 175,
                     'resource':'n_cubicles',
                     'label': "Being Treated"},

                    {'event': 'depart',
                     'x':  270, 'y': 70,
                     'label': "Exit"}

                ])


# Generate animation using the step-by-step functions 

Using the three step-by-step functions allows us to intervene in the produced dataframe and manually take control of the icons in use.

This will allow us to show the high-priority patients with a unique icon so we can see their frequency and how they are handled in the final model. 

In [ ]:
full_patient_df_plus_pos = generate_animation_df(
    full_entity_df=full_patient_df,
    event_position_df=event_position_df,
    entity_col_name="patient",
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    gap_between_entities=10,
    gap_between_resources=10,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    debug_mode=True
    )

full_patient_df_plus_pos.sort_values(['patient', 'snapshot_time']).head(15)

In [ ]:
def show_priority_icon(row):
            if "more" not in row["icon"]:
                if row["pathway"] == 1:
                        return "🚨"
                else:
                    return row["icon"]
            else:
                return row["icon"]

In [ ]:
full_patient_df_plus_pos = full_patient_df_plus_pos.assign(
            icon=full_patient_df_plus_pos.apply(show_priority_icon, axis=1)
            )

In [ ]:
full_patient_df_plus_pos.head(15)

In [ ]:
generate_animation(
        full_entity_df_plus_pos=full_patient_df_plus_pos.sort_values(['patient', 'snapshot_time']),
        event_position_df= event_position_df,
        scenario=g(),
        entity_col_name="patient",
        debug_mode=True,
        setup_mode=False,
        include_play_button=True,
        entity_icon_size=16,
        resource_icon_size=20,
        gap_between_resource_rows=30,
        plotly_height=600,
        frame_duration=800,
        frame_transition_duration=200,
        plotly_width=1000,
        override_x_max=300,
        override_y_max=500,
        time_display_units="dhm",
        display_stage_labels=False,
        add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_1_simplest_case/Simplest%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
    )

# Rerun, but using the all-in-one animation function (which will not show different priority icons)

In [ ]:
animate_activity_log(
        event_log=my_trial.all_event_logs[my_trial.all_event_logs['run']==1],
        event_position_df= event_position_df,
        scenario=g(),
        entity_col_name="patient",
        debug_mode=True,
        setup_mode=False,
        every_x_time_units=1,
        include_play_button=True,
        entity_icon_size=16,
        resource_icon_size=20,
        gap_between_entities=6,
        gap_between_queue_rows=25,
        gap_between_resource_rows=25,
        plotly_height=600,
        frame_duration=200,
        plotly_width=1000,
        override_x_max=300,
        override_y_max=500,
        limit_duration=g.sim_duration,
        wrap_queues_at=25,
        step_snapshot_max=125,
        time_display_units="dhm",
        display_stage_labels=False,
        add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_1_simplest_case/Simplest%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
    )